# Computational Imaging - Group I Project: Superresolution
**Dataset**: LSUN Church (256x256)
**Methods**: UNet (End-to-end), Total Variation (Variational), DiffPIR (Generative)


## 1. Setup and Installs
Install required packages. Diffusers is needed for DiffPIR.


In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import os
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from diffusers import UNet2DModel
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")

In [ ]:
## 2. Normalization and Metrics

class ImageNormalizer:
    def __init__(self, mode='0_1'):
        self.mode = mode

    def normalize(self, tensor: torch.Tensor) -> torch.Tensor:
        if tensor.max() > 1.0:
            tensor = tensor.float() / 255.0
        if self.mode == '0_1':
            return tensor
        elif self.mode == 'neg1_1':
            return (tensor * 2.0) - 1.0
        else:
            raise ValueError("Mode not supported")

    def denormalize(self, tensor: torch.Tensor) -> torch.Tensor:
        if self.mode == '0_1':
            return torch.clamp(tensor, 0.0, 1.0)
        elif self.mode == 'neg1_1':
            return torch.clamp((tensor + 1.0) / 2.0, 0.0, 1.0)

default_normalizer = ImageNormalizer(mode='0_1')

def compute_psnr(pred, target):
    pred_np = pred.detach().cpu().permute(1, 2, 0).numpy() if pred.ndim == 3 else pred.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    target_np = target.detach().cpu().permute(1, 2, 0).numpy() if target.ndim == 3 else target.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    return float(psnr_metric(target_np, pred_np, data_range=1.0))

def compute_ssim(pred, target):
    pred_np = pred.detach().cpu().permute(1, 2, 0).numpy() if pred.ndim == 3 else pred.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    target_np = target.detach().cpu().permute(1, 2, 0).numpy() if target.ndim == 3 else target.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    return float(ssim_metric(target_np, pred_np, data_range=1.0, channel_axis=2))

In [ ]:
## 3. Degradation Operator and Dataset

class DegradationOperator:
    def __init__(self, sr_factor=2, noise_std=0.005, seed=42, device='cpu'):
        self.sr_factor = sr_factor
        self.noise_std = noise_std
        self.seed = seed
        self.device = device
        
    def __call__(self, img_tensor):
        if isinstance(img_tensor, Image.Image):
            img_tensor = torch.from_numpy(np.array(img_tensor)).float()
            if len(img_tensor.shape) == 3: img_tensor = img_tensor.permute(2, 0, 1)
        img_tensor = default_normalizer.normalize(img_tensor).float().to(self.device)
        if len(img_tensor.shape) == 3: img_tensor = img_tensor.unsqueeze(0)
        
        degraded = self._downsample(img_tensor)
        degraded = self._add_noise(degraded)
        return degraded.squeeze(0)
    
    def _downsample(self, x):
        kernel_size = self.sr_factor * 2 + 1
        sigma = self.sr_factor / np.sqrt(8 * np.log(2))
        kernel = self._create_gaussian_kernel(kernel_size, sigma).to(self.device)
        B, C, H, W = x.shape
        blurred = torch.zeros_like(x)
        for i in range(C):
            blurred[:, i:i+1, :, :] = F.conv2d(x[:, i:i+1, :, :], kernel, padding=kernel_size // 2)
        return F.interpolate(blurred, scale_factor=1/self.sr_factor, mode='bilinear', align_corners=False)
    
    def _add_noise(self, x):
        torch.manual_seed(self.seed)
        return torch.clamp(x + torch.randn_like(x) * self.noise_std, 0.0, 1.0)
    
    @staticmethod
    def _create_gaussian_kernel(kernel_size, sigma):
        x = torch.arange(kernel_size).float() - (kernel_size - 1) / 2.0
        gauss = torch.exp(-(x ** 2) / (2 * sigma ** 2))
        kernel_1d = gauss / gauss.sum()
        kernel_2d = kernel_1d.unsqueeze(-1) @ kernel_1d.unsqueeze(0)
        return kernel_2d.unsqueeze(0).unsqueeze(0)

import random

class SRDataset(Dataset):
    def __init__(self, image_dir, sr_factor=None, noise_std=None, max_images=None):
        self.files = sorted(Path(image_dir).glob("*.png")) + sorted(Path(image_dir).glob("*.jpg"))
        if max_images: self.files = self.files[:max_images]
        self.sr_factors = [sr_factor] if sr_factor is not None else [2, 4]
        self.noise_stds = [noise_std] if noise_std is not None else [0.005, 0.01]
        
        self.is_blind = (sr_factor is None or noise_std is None)
        if not self.is_blind:
            self.degrader = DegradationOperator(sr_factor=sr_factor, noise_std=noise_std)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        # FORZIAMO IL RESIZE A 256x256 INDIPENDENTEMENTE DAL DATASET SCELTO!
        if img.size != (256, 256):
            img = img.resize((256, 256), Image.Resampling.BICUBIC)
            
        gt = default_normalizer.normalize(torch.from_numpy(np.array(img)).float().permute(2, 0, 1))
        
        if self.is_blind:
            sr = random.choice(self.sr_factors)
            noise = random.choice(self.noise_stds)
            degrader = DegradationOperator(sr_factor=sr, noise_std=noise)
            degrader.seed = random.randint(0, 100000)
        else:
            degrader = self.degrader
            degrader.seed = 42 + idx
            
        degraded = degrader(img)
        degraded_up = F.interpolate(degraded.unsqueeze(0), size=(256, 256), mode='bicubic', align_corners=False).squeeze(0)
        return torch.clamp(degraded_up, 0.0, 1.0), gt

# KAGGLE CONFIGURATION
# Set the path to your dataset correctly. Es: "/kaggle/input/lsun-church-data/kaggle/working/lsun_church"
KAGGLE_DATASET_PATH = "/kaggle/input/datasets/mattiafurini/lsun-church-256x256"

In [ ]:
## 4. End-to-End: UNet Architecture & Training Loop

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoder_blocks = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        prev = in_channels
        for feat in features:
            self.encoder_blocks.append(DoubleConv(prev, feat))
            prev = feat
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        self.upconvs = nn.ModuleList()
        self.decoder_blocks = nn.ModuleList()
        prev = features[-1] * 2
        for feat in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(prev, feat, kernel_size=2, stride=2))
            self.decoder_blocks.append(DoubleConv(feat * 2, feat))
            prev = feat
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for enc in self.encoder_blocks:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for upconv, dec, skip in zip(self.upconvs, self.decoder_blocks, skips):
            x = upconv(x)
            if x.shape != skip.shape:
                x = F.pad(x, [0, skip.shape[3]-x.shape[3], 0, skip.shape[2]-x.shape[2]])
            x = torch.cat([skip, x], dim=1)
            x = dec(x)
        return torch.sigmoid(self.final_conv(x))

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss, psnr_vals, ssim_vals = 0.0, [], []
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        total_loss += criterion(outputs, targets).item()
        for i in range(outputs.shape[0]):
            psnr_vals.append(compute_psnr(outputs[i], targets[i]))
            ssim_vals.append(compute_ssim(outputs[i], targets[i]))
    return total_loss / len(dataloader), np.mean(psnr_vals), np.mean(ssim_vals)

In [ ]:
## 5. Variational: Total Variation (TV)

class TVSuperResolution:
    def __init__(self, sr_factor=2, noise_std=0.005, lambda_tv=0.01, n_iter=200, step_size=0.005, device='cpu'):
        self.sr_factor, self.lambda_tv, self.n_iter = sr_factor, lambda_tv, n_iter
        self.step_size, self.device = step_size, device
        k_size = sr_factor * 2 + 1
        self.kernel = DegradationOperator._create_gaussian_kernel(k_size, sr_factor/np.sqrt(8*np.log(2))).to(device)
        self.padding = k_size // 2

    def solve(self, y):
        C, H_lr, W_lr = y.shape[1] if y.ndim==4 else y.shape[0], y.shape[-2], y.shape[-1]
        H_hr, W_hr = H_lr * self.sr_factor, W_lr * self.sr_factor
        if y.ndim==3: y = y.unsqueeze(0)
        x = torch.clamp(F.interpolate(y, size=(H_hr, W_hr), mode='bicubic', align_corners=False), 0, 1)
        z, t = x.clone(), 1.0
        
        for _ in range(self.n_iter):
            x_old = x.clone()
            Hz = F.interpolate(torch.cat([F.conv2d(z[:, c:c+1], self.kernel, padding=self.padding) for c in range(C)], 1), scale_factor=1/self.sr_factor, mode='bilinear', align_corners=False)
            res = Hz - y
            up = F.interpolate(res, size=(H_hr, W_hr), mode='bilinear', align_corners=False)
            grad_data = torch.cat([F.conv2d(up[:, c:c+1], self.kernel, padding=self.padding) for c in range(C)], 1)
            
            dx = torch.zeros_like(z); dx[:, :, :, :-1] = z[:, :, :, 1:] - z[:, :, :, :-1]
            dy = torch.zeros_like(z); dy[:, :, :-1, :] = z[:, :, 1:, :] - z[:, :, :-1, :]
            mag = torch.sqrt(dx**2 + dy**2 + 1e-6)
            nx, ny = dx/mag, dy/mag
            div = torch.zeros_like(z)
            div[:,:,:,0] = nx[:,:,:,0]; div[:,:,:,1:-1] = nx[:,:,:,1:-1]-nx[:,:,:,:-2]; div[:,:,:,-1] = -nx[:,:,:,-2]
            div[:,:,0,:] += ny[:,:,0,:]; div[:,:,1:-1,:] += ny[:,:,1:-1,:]-ny[:,:,:-2,:]; div[:,:,-1,:] += -ny[:,:,:-2,:]
            
            x = torch.clamp(z - self.step_size * (grad_data - self.lambda_tv * div), 0, 1)
            t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
            z = torch.clamp(x + ((t - 1) / t_new) * (x - x_old), 0, 1)
            t = t_new
        return x.squeeze(0)

In [ ]:
## 6. Generative: DiffPIR
## > **⚠️ WARNING on DiffPIR Execution Time:** 
## > Diffusion models are slow during inference (e.g., 100 steps per image). Running DiffPIR on the full dataset of 4000 images could take several hours. 
## > It is strongly recommended to evaluate DiffPIR on a small subset (e.g., 50-100 images) to generate visual comparisons and compute metrics for the presentation.

def get_beta_schedule(steps=1000, start=0.0001, end=0.02):
    return np.linspace(start, end, steps, dtype=np.float64)

class DiffusersUNetWrapper(nn.Module):
    def __init__(self, unet):
        super().__init__()
        self.unet = unet
    def forward(self, x, t): return self.unet(x, t).sample

class DiffPIR:
    def __init__(self, sr_factor=2, noise_std=0.005, sampling_steps=100, rho=1.0, lambda_reg=0.5, device='cuda', pretrained_unet=None):
        self.sr_factor, self.rho, self.lambda_reg, self.device = sr_factor, rho, lambda_reg, device
        betas = get_beta_schedule()
        alphas = 1.0 - betas
        self.alphas_cumprod = np.cumprod(alphas)
        self.t_seq = np.linspace(0, 999, sampling_steps, dtype=int)[::-1]
        k_size = sr_factor * 2 + 1
        self.kernel = DegradationOperator._create_gaussian_kernel(k_size, sr_factor/np.sqrt(8*np.log(2))).to(device)
        self.padding = k_size // 2
        
        print("Loading ddpm-ema-church-256 model...")
        unet = pretrained_unet if pretrained_unet is not None else UNet2DModel.from_pretrained("google/ddpm-ema-church-256").to(device)
        self.model = DiffusersUNetWrapper(unet).eval()

    @torch.no_grad()
    def restore(self, degraded_img):
        y = degraded_img.unsqueeze(0).to(self.device) * 2 - 1 if degraded_img.dim() == 3 else degraded_img.to(self.device) * 2 - 1
        C, H_hr, W_hr = y.shape[1], y.shape[2]*self.sr_factor, y.shape[3]*self.sr_factor
        x_t = torch.randn(1, C, H_hr, W_hr, device=self.device)
        
        for i, t in enumerate(self.t_seq):
            t_tens = torch.full((1,), t, device=self.device, dtype=torch.long)
            noise_pred = self.model(x_t, t_tens)
            
            alpha_c = self.alphas_cumprod[t]
            x0_pred = (x_t - np.sqrt(1 - alpha_c) * noise_pred) / np.sqrt(alpha_c)
            x0_pred = torch.clamp(x0_pred, -1.0, 1.0)
            
            rho_t = self.rho * self.lambda_reg / (self.lambda_reg + (1-alpha_c))
            
            # Forward H
            blurred = torch.cat([F.conv2d(x0_pred[:, c:c+1], self.kernel, padding=self.padding) for c in range(C)], 1)
            Hx0 = F.interpolate(blurred, scale_factor=1/self.sr_factor, mode='bilinear', align_corners=False)
            res = Hx0 - y
            # Adjoint H^T
            up = F.interpolate(res, size=(H_hr, W_hr), mode='bilinear', align_corners=False)
            grad_data = torch.cat([F.conv2d(up[:, c:c+1], self.kernel, padding=self.padding) for c in range(C)], 1)
            
            x0_corr = torch.clamp(x0_pred - rho_t * grad_data, -1.0, 1.0)
            
            if i < len(self.t_seq) - 1:
                alpha_c_prev = self.alphas_cumprod[self.t_seq[i+1]]
                if self.t_seq[i+1] > 0:
                    implicit_noise = (x_t - np.sqrt(alpha_c)*x0_corr) / np.sqrt(1-alpha_c)
                    x_t = np.sqrt(alpha_c_prev)*x0_corr + np.sqrt(1-alpha_c_prev)*implicit_noise
                else: x_t = x0_corr
            else: x_t = x0_corr
            
        return ((x_t.squeeze(0) + 1.0) / 2.0).clamp(0.0, 1.0)

In [ ]:
## 7. UNet Training Execution

SR_FACTOR = 2
NOISE_STD = 0.005
EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-3

# Initialize Model
unet_model = UNet(in_channels=3, out_channels=3).to(device)

# Parallelize if multiple GPUs are available (Kaggle T4 x2)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    unet_model = nn.DataParallel(unet_model)

# Training Setup
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(unet_model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)


print("--- CONTROLLO DATASET E IMMAGINI LOCALI ---")
# Usiamo direttamente le cartelle già divise!
train_dataset = SRDataset(f"{KAGGLE_DATASET_PATH}/train")
val_dataset = SRDataset(f"{KAGGLE_DATASET_PATH}/val")

if len(train_dataset) == 0:
    raise ValueError("Nessuna immagine trovata in train! Controlla il KAGGLE_DATASET_PATH")

test_img = Image.open(train_dataset.files[0])
print(f"Risoluzione originale immagini caricate: {test_img.size}")
if test_img.size != (256, 256):
    print("-> Nessun problema: SRDataset farà il RESIZE AUTOMATICO a 256x256.")
else:
    print("-> ✅ PERFETTO: Le immagini sono già nativamente 256x256!")

print(f"Immagini caricate: {len(train_dataset)} Training | {len(val_dataset)} Validation.")
print("------------------------------------\n")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

best_psnr = 0.0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(unet_model, train_loader, optimizer, criterion, device)
    val_loss, val_psnr, val_ssim = validate(unet_model, val_loader, criterion, device)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | "
          f"PSNR: {val_psnr:.2f} dB | SSIM: {val_ssim:.4f} | Time: {time.time()-t0:.1f}s")
    
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(unet_model.state_dict(), "unet_best.pth")
        print(f"  -> Best model saved! (PSNR: {best_psnr:.2f})")

print("UNet setup complete.")

In [ ]:
class TVSuperResolution:
    def __init__(self, lambda_tv=0.01, n_iter=200, step_size=0.005, device='cpu'):
        self.lambda_tv, self.n_iter = lambda_tv, n_iter
        self.step_size, self.device = step_size, device

    def solve(self, y, sr_factor=2):
        # Creiamo il kernel in base al fattore di Super-Resolution
        k_size = sr_factor * 2 + 1
        sigma = sr_factor / np.sqrt(8 * np.log(2))
        kernel = DegradationOperator._create_gaussian_kernel(k_size, sigma).to(self.device)
        padding = k_size // 2

        C, H_lr, W_lr = y.shape[1] if y.ndim==4 else y.shape[0], y.shape[-2], y.shape[-1]
        H_hr, W_hr = H_lr * sr_factor, W_lr * sr_factor
        if y.ndim==3: y = y.unsqueeze(0)
        
        # Inizializzazione con interpolazione bicubica
        x = torch.clamp(F.interpolate(y, size=(H_hr, W_hr), mode='bicubic', align_corners=False), 0, 1)
        z, t = x.clone(), 1.0
        
        for _ in range(self.n_iter):
            x_old = x.clone()
            
            # --- Data Fidelity (Gradiente) ---
            blurred = torch.cat([F.conv2d(z[:, c:c+1], kernel, padding=padding) for c in range(C)], 1)
            Hz = F.interpolate(blurred, scale_factor=1/sr_factor, mode='bilinear', align_corners=False)
            res = Hz - y
            up = F.interpolate(res, size=(H_hr, W_hr), mode='bilinear', align_corners=False)
            grad_data = torch.cat([F.conv2d(up[:, c:c+1], kernel, padding=padding) for c in range(C)], 1)
            
            # --- Total Variation (Divergenza) ---
            dx = torch.zeros_like(z)
            dx[:, :, :, :-1] = z[:, :, :, 1:] - z[:, :, :, :-1]
            dy = torch.zeros_like(z)
            dy[:, :, :-1, :] = z[:, :, 1:, :] - z[:, :, :-1, :]
            
            mag = torch.sqrt(dx**2 + dy**2 + 1e-6)
            nx, ny = dx/mag, dy/mag
            
            div_x = torch.zeros_like(z)
            div_x[:,:,:,0] = nx[:,:,:,0]
            div_x[:,:,:,1:-1] = nx[:,:,:,1:-1] - nx[:,:,:,:-2]
            div_x[:,:,:,-1] = -nx[:,:,:,-2]
            
            div_y = torch.zeros_like(z)
            div_y[:,:,0,:] = ny[:,:,0,:]
            div_y[:,:,1:-1,:] = ny[:,:,1:-1,:] - ny[:,:,:-2,:]
            div_y[:,:,-1,:] = -ny[:,:,-2,:]
            
            div = div_x + div_y  # Somma sicura senza in-place!
            
            # --- Aggiornamento FISTA ---
            x = torch.clamp(z - self.step_size * (grad_data - self.lambda_tv * div), 0, 1)
            t_new = (1 + np.sqrt(1 + 4*t**2)) / 2
            z = torch.clamp(x + ((t - 1) / t_new) * (x - x_old), 0, 1)
            t = t_new
            
        return x.squeeze(0)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import os

print("Inizializzazione Solutori in corso...")

if 'SR_FACTOR' not in locals():
    SR_FACTOR = 2
if 'NOISE_STD' not in locals():
    NOISE_STD = 0.005

# 1. Carica il Test Set
test_dataset = SRDataset(f"{KAGGLE_DATASET_PATH}/test", sr_factor=SR_FACTOR, noise_std=NOISE_STD)

# 2. Ricarica la UNet
if 'unet_model' not in locals():
    unet_model = UNet(in_channels=3, out_channels=3).to(device)
    if torch.cuda.device_count() > 1:
        unet_model = nn.DataParallel(unet_model)

if os.path.exists('unet_best.pth'):
    weights_path = 'unet_best.pth'
elif os.path.exists('weights/unet_best.pth'):
    weights_path = 'weights/unet_best.pth'
elif os.path.exists('/kaggle/input/datasets/mattiafurini/weights/unet_best.pth'):
    weights_path = '/kaggle/input/datasets/mattiafurini/weights/unet_best.pth'
else:
    weights_path = '/kaggle/input/datasets/mattiafurini/weights/unet_best.pth' 
    print("⚠️ Attenzione: Assicurati che unet_best.pth sia nel path specificato!")

unet_model.load_state_dict(torch.load(weights_path, map_location=device))
unet_model.eval()
print(f"✅ Pesi UNet caricati da {weights_path}")

# 3. Prepara i solutori TV e DiffPIR
tv_solver = TVSuperResolution(lambda_tv=0.05, n_iter=300, device=device)
diffpir_solver = DiffPIR(device=device)

# Ricreiamo il degrader per ottenere la VERA immagine a bassa risoluzione per TV e DiffPIR
degrader = DegradationOperator(sr_factor=SR_FACTOR, noise_std=NOISE_STD, device=device)

# --- VALUTAZIONE VISIVA SU 3 IMMAGINI ---
print("\nGenerazione del Plot Comparativo...")
fig, axes = plt.subplots(3, 5, figsize=(22, 13))
col_titles = ['Originale (GT)', 'Degradata (Bassa Risp.)', 'UNet (End-to-End)', 'TV (Variazionale)', 'DiffPIR (Generativo)']

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=16, fontweight='bold')

import random; fixed_indices_7 = random.sample(range(len(test_dataset)), 3)
for row_idx, i in enumerate(fixed_indices_7):
    print(f"Ricostruzione Immagine {row_idx+1}/3 in corso...")
    
    degraded_up, gt = test_dataset[i]
    
    degrader.seed = 42 + i
    degraded_lr = degrader(gt.to(device))
    
    with torch.no_grad():
        unet_out = unet_model(degraded_up.unsqueeze(0).to(device)).squeeze(0).cpu()
    
    tv_out = tv_solver.solve(degraded_lr).cpu()
    diff_out = diffpir_solver.restore(degraded_lr).cpu()
    
    imgs = [gt, degraded_up, unet_out, tv_out, diff_out]
    for j, img in enumerate(imgs):
        img_np = img.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np, 0, 1)
        axes[row_idx, j].imshow(img_np)
        axes[row_idx, j].axis('off')

plt.tight_layout()
plt.savefig("visual_evaluation_section7.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Comprehensive Evaluation (4 Degradation Scenarios)
Come richiesto dalle specifiche di progetto, valutiamo tutti e tre i modelli (UNet, TV, DiffPIR) sulle 4 combinazioni di degradazione:
- **Scenari:** SR = {2, 4} $\times$ Noise = {0.005, 0.01}

Poiché DiffPIR è molto costoso computazionalmente e UNet non generalizza bene a degradazioni unseen (è stata addestrata per un singolo scenario), valuteremo un subset del Test Set (es. `NUM_TEST_IMAGES = 10` o 50). I risultati verranno poi formattati in una tabella pandas.

In [ ]:
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
scenarios = [
    {'sr': 2, 'noise': 0.005},
    {'sr': 2, 'noise': 0.01},
    {'sr': 4, 'noise': 0.005},
    {'sr': 4, 'noise': 0.01}
]
NUM_TEST_IMAGES = 10  # Aumentare se si ha tempo computazionale a disposizione

results = {
    'Method': ['UNet', 'TV', 'DiffPIR']
}

unet_model.eval()

print("\nLoading Diffusion Model ONCE to save memory...")
from diffusers import UNet2DModel
pretrained_unet = UNet2DModel.from_pretrained("google/ddpm-ema-church-256").to(device)

for scenario in scenarios:
    sr = scenario['sr']
    noise = scenario['noise']
    col_name = f"SR={sr}, n={noise}"
    print(f"\n--- Inizio test per {col_name} ---")
    
    # Crea un dataloader ad-hoc per questo scenario
    scenario_dataset = SRDataset(f"{KAGGLE_DATASET_PATH}/test", sr_factor=sr, noise_std=noise)
    
    # Inizializza/Aggiorna i solver che dipendono da H (TV e DiffPIR)
    tv_solver = TVSuperResolution(lambda_tv=0.05, n_iter=300, device=device)
    diffpir_solver = DiffPIR(sr_factor=sr, noise_std=noise, sampling_steps=50, device=device, pretrained_unet=pretrained_unet)
    
    unet_psnr_list, unet_ssim_list = [], []
    tv_psnr_list, tv_ssim_list = [], []
    diff_psnr_list, diff_ssim_list = [], []
    
    for i in range(min(NUM_TEST_IMAGES, len(scenario_dataset))):
        degraded, gt = scenario_dataset[i]
        
        # SRDataset restituisce sempre l'immagine GIA' scalata a 256x256 (degraded_up)
        degraded_up = degraded
        y_up = degraded_up.unsqueeze(0).to(device)
        x_gt = gt.unsqueeze(0).to(device)
        
        # Ricreiamo la vera LR observation per i metodi matematici
        scenario_dataset.degrader.seed = 42 + i
        y_lr = scenario_dataset.degrader(gt.unsqueeze(0).to(device)).to(device)
        
        # 1. UNet (prende 256x256)
        with torch.no_grad():
            x_unet = unet_model(y_up)
            
        # 2. TV (prende l'immagine LR piccola)
        x_tv = tv_solver.solve(y_lr, sr_factor=sr)
        
        # 3. DiffPIR (prende l'immagine LR piccola)
        x_diff = diffpir_solver.restore(y_lr)
        
        # Calcolo Metriche
        print(f'x_unet: {x_unet.shape}, x_gt: {x_gt.shape}')
        unet_psnr_list.append(compute_psnr(x_unet, x_gt))
        unet_ssim_list.append(compute_ssim(x_unet, x_gt))
        
        tv_psnr_list.append(compute_psnr(x_tv, x_gt))
        tv_ssim_list.append(compute_ssim(x_tv, x_gt))
        
        diff_psnr_list.append(compute_psnr(x_diff, x_gt))
        diff_ssim_list.append(compute_ssim(x_diff, x_gt))
        
        if (i+1) % 5 == 0:
            print(f"[{i+1}/{NUM_TEST_IMAGES}] processate.")

    # Media
    res_unet = f"{np.mean(unet_psnr_list):.2f} / {np.mean(unet_ssim_list):.4f}"
    res_tv   = f"{np.mean(tv_psnr_list):.2f} / {np.mean(tv_ssim_list):.4f}"
    res_diff = f"{np.mean(diff_psnr_list):.2f} / {np.mean(diff_ssim_list):.4f}"
    
    results[col_name] = [res_unet, res_tv, res_diff]

# Costruzione DataFrame Finale
df_results = pd.DataFrame(results)
print("\n=======================================================")
print("TABELLA RISULTATI FINALE (Da copiare nella Slide 14)")
print("=======================================================")
display(df_results)

df_results.to_csv("risultati_metriche.csv", index=False)
df_results.to_markdown("risultati_metriche.md", index=False)
print("\nTabella salvata in risultati_metriche.csv e risultati_metriche.md")


## 9. Qualitative Visual Breakdown (Severe Degradation)
Come mostrato in Slide 16, eseguiamo un test visivo sull'ultimo scenario (SR=4, Noise=0.01) per evidenziare le differenze strutturali: collasso della UNet vs Edge recovery del TV vs Allucinazioni di DiffPIR.

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

# Riusiamo l'ultimo scenario testato (SR=4, Noise=0.01) dal blocco precedente
print("Generazione griglia visiva per SR=4, Noise=0.01...")
NUM_VISUAL_TEST = 3

fig, axes = plt.subplots(NUM_VISUAL_TEST, 5, figsize=(20, 4 * NUM_VISUAL_TEST))
cols = ['Original (GT)', 'Degraded (SR=4, n=0.01)', 'UNet (Collasso)', 'TV (Watercolor)', 'DiffPIR (Allucinazioni)']
for ax, col in zip(axes[0], cols):
    ax.set_title(col, fontsize=14, fontweight='bold')

import random; fixed_indices_9 = random.sample(range(len(scenario_dataset)), 3)
for row_idx, i in enumerate(fixed_indices_9):
    # Prendi alcune immagini
    degraded_up, gt = scenario_dataset[i]
    
    y_up = degraded_up.unsqueeze(0).to(device)
    
    scenario_dataset.degrader.seed = 42 + i
    y_lr = scenario_dataset.degrader(gt.unsqueeze(0).to(device)).to(device)
    
    with torch.no_grad():
        x_unet = unet_model(y_up)
        
    x_tv = tv_solver.solve(y_lr, sr_factor=sr)
    x_diff = diffpir_solver.restore(y_lr)
    
    if x_unet.dim() == 3: x_unet = x_unet.unsqueeze(0)
    if x_tv.dim() == 3: x_tv = x_tv.unsqueeze(0)
    if x_diff.dim() == 3: x_diff = x_diff.unsqueeze(0)
    
    # Plot
    axes[row_idx, 0].imshow(gt.permute(1, 2, 0).cpu().numpy())
    axes[row_idx, 0].axis('off')
    
    axes[row_idx, 1].imshow(y_up[0].permute(1, 2, 0).cpu().numpy())
    axes[row_idx, 1].axis('off')
    
    axes[row_idx, 2].imshow(x_unet[0].clamp(0,1).permute(1, 2, 0).cpu().numpy())
    axes[row_idx, 2].axis('off')
    
    axes[row_idx, 3].imshow(x_tv[0].clamp(0,1).permute(1, 2, 0).cpu().numpy())
    axes[row_idx, 3].axis('off')
    
    axes[row_idx, 4].imshow(x_diff[0].clamp(0,1).permute(1, 2, 0).cpu().numpy())
    axes[row_idx, 4].axis('off')

plt.tight_layout()
plt.savefig("visual_evaluation_section9_severe.png", dpi=150, bbox_inches='tight')
plt.show()


## 10. Generazione Immagini per Presentazione
Questa cella seleziona immagini casuali dal test set e genera le 3 immagini richieste dalla presentazione:
1. `degradation_grid_image.png`: Griglia 1x4 con le 4 degradazioni applicate a una singola immagine.
2. `unet_zoom_smooth_bricks.png`: Immagine intera per mostrare l'effetto di eccessivo smoothing della UNet.
3. `kaggl_3x5_grid_1.png`: Confronto visivo finale (Original, Degraded, UNet, TV, DiffPIR) su 3 immagini casuali (scenario SR=4, Noise=0.01).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import os

print("--- Generazione Immagini per Presentazione ---")

if 'device' not in locals(): device = 'cuda' if torch.cuda.is_available() else 'cpu'
if 'test_dataset' not in locals(): 
    test_dataset = SRDataset(f"{KAGGLE_DATASET_PATH}/test", sr_factor=2, noise_std=0.005)

# Selezioniamo indici fissi per usare sempre le stesse immagini in tutti i plot
import random; fixed_indices = random.sample(range(len(test_dataset)), 3)  # Immagini random del test set
idx_1 = fixed_indices[0]
_, gt_img_1 = test_dataset[idx_1]

# ---------------------------------------------------------
# 1. Griglia Degradazione (2x2) - degradation_grid_image.png
# ---------------------------------------------------------
print("Generazione 1: Griglia Degradazione...")
fig_1, axes_1 = plt.subplots(1, 4, figsize=(20, 5))
scenarios = [
    (2, 0.005, 0), (2, 0.01, 1),
    (4, 0.005, 2), (4, 0.01, 3)
]

for sr, noise, c in scenarios:
    deg_op = DegradationOperator(sr_factor=sr, noise_std=noise, device=device)
    deg_op.seed = 42 + idx_1
    y_lr = deg_op(gt_img_1.to(device)).cpu()
    
    # Resize a 256 per uniformità visiva
    y_up = torch.nn.functional.interpolate(y_lr.unsqueeze(0), size=(256, 256), mode='bicubic', align_corners=False).squeeze(0)
    
    img_np = np.clip(y_up.permute(1, 2, 0).numpy(), 0, 1)
    axes_1[c].imshow(img_np)
    axes_1[c].set_title(f"SR={sr}, Noise={noise}", fontsize=14)
    axes_1[c].axis("off")

plt.tight_layout(pad=1.5)
plt.savefig("degradation_grid_image.png", dpi=150, bbox_inches='tight', pad_inches=0.1)
plt.close(fig_1)
print("Salvata: degradation_grid_image.png")

# ---------------------------------------------------------
# 2. Griglia Comparativa 3x5 - sr=2 e sr=4
# ---------------------------------------------------------
from diffusers import UNet2DModel
diff_unet = UNet2DModel.from_pretrained("google/ddpm-ema-church-256").to(device)

def generate_3x5_grid(sr_vis, noise_vis, filename):
    print(f"Generazione Griglia 3x5 per SR={sr_vis}, n={noise_vis}...")
    diffpir_solver_vis = DiffPIR(sr_factor=sr_vis, noise_std=noise_vis, sampling_steps=50, device=device, pretrained_unet=diff_unet)
    tv_solver_vis = TVSuperResolution(lambda_tv=0.05, n_iter=300, device=device)
    deg_op_vis = DegradationOperator(sr_factor=sr_vis, noise_std=noise_vis, device=device)
    
    num_vis = 3
    fig_3, axes_3 = plt.subplots(num_vis, 5, figsize=(20, 4 * num_vis))
    cols = ['Original (GT)', f'Degraded (SR={sr_vis}, n={noise_vis})', 'UNet', 'TV', 'DiffPIR']
    for ax, col in zip(axes_3[0], cols):
        ax.set_title(col, fontsize=14, fontweight='bold')
    
    for i, idx_3 in enumerate(fixed_indices[:num_vis]):
        _, gt_img_3 = test_dataset[idx_3]
        deg_op_vis.seed = 42 + idx_3
        y_lr_3 = deg_op_vis(gt_img_3.to(device)).to(device)
        y_up_3 = torch.nn.functional.interpolate(y_lr_3.unsqueeze(0), size=(256, 256), mode='bicubic', align_corners=False)
        
        with torch.no_grad():
            unet_out_3 = unet_model(y_up_3).squeeze(0).cpu()
        
        tv_out_3 = tv_solver_vis.solve(y_lr_3, sr_factor=sr_vis).cpu()
        diff_out_3 = diffpir_solver_vis.restore(y_lr_3).cpu()
        
        imgs = [gt_img_3.cpu(), y_up_3.squeeze(0).cpu(), unet_out_3, tv_out_3, diff_out_3]
        for j, img in enumerate(imgs):
            axes_3[i, j].imshow(np.clip(img.permute(1, 2, 0).numpy(), 0, 1))
            axes_3[i, j].axis('off')

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig_3)
    print(f"Salvata: {filename}")

generate_3x5_grid(2, 0.005, "kaggl_3x5_grid_sr2_n0.005.png")
generate_3x5_grid(2, 0.01, "kaggl_3x5_grid_sr2_n0.01.png")
generate_3x5_grid(4, 0.005, "kaggl_3x5_grid_sr4_n0.005.png")
generate_3x5_grid(4, 0.01, "severe_degradation_breakdown.png")

# ---------------------------------------------------------
# 3. Immagini per i Modelli (UNet, TV, DiffPIR) su SR=4 (Intere, No Zoom)
# ---------------------------------------------------------
print("Generazione Immagini intere per SR=4...")
sr_severe = 4
noise_severe = 0.01

dataset_severe = SRDataset(f"{KAGGLE_DATASET_PATH}/test", sr_factor=sr_severe, noise_std=noise_severe)
tv_solver_severe = TVSuperResolution(lambda_tv=0.05, n_iter=300, device=device)
diffpir_solver_severe = DiffPIR(sr_factor=sr_severe, noise_std=noise_severe, sampling_steps=50, device=device, pretrained_unet=diff_unet)

img_idx = fixed_indices[0]
degraded_up, gt = dataset_severe[img_idx]

dataset_severe.degrader.seed = 42 + img_idx
y_lr = dataset_severe.degrader(gt.unsqueeze(0).to(device)).to(device)

print("Esecuzione Modelli in corso su immagine intera (attendi DiffPIR)...")
with torch.no_grad():
    unet_out = unet_model(degraded_up.unsqueeze(0).to(device)).squeeze(0).cpu()

tv_out = tv_solver_severe.solve(y_lr, sr_factor=sr_severe).cpu()
diff_out = diffpir_solver_severe.restore(y_lr).cpu()

def save_image(img_tensor, filename):
    img_np = np.clip(img_tensor.permute(1, 2, 0).numpy(), 0, 1)
    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(img_np)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Salvata: {filename}")

save_image(unet_out, "unet_zoom_smooth_bricks.png")
save_image(tv_out, "tv_zoom_watercolor.png")
save_image(diff_out, "diffpir_zoom_hallucinations.png")

print("--- TUTTE LE IMMAGINI GENERATE CON SUCCESSO ---")


## 11. Empirical Test: Hallucination vs Memorization
Questo test verifica empiricamente se gli artefatti generati da DiffPIR sotto *severe degradation* siano dovuti a pura Hallucination (libertà generativa del modello dovuta a un vincolo data-consistency debole) oppure a Memorizzazione (overfitting del training set).

Il test passa la **stessa identica immagine degradata** a DiffPIR per 3 volte, cambiando solo il *seed* iniziale della diffusione. 
- Se otteniamo 3 chiese strutturalmente diverse, è confermata l'ipotesi di pura **Hallucination**.
- Se otteniamo la stessa chiesa fantasma, è confermata la **Memorizzazione/Overfitting**.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print("--- Esecuzione Test Empirico: Hallucination vs Memorization ---")

# Scegliamo un'immagine e applichiamo severe degradation (SR=4, Noise=0.01)
sr_emp, noise_emp = 4, 0.01
test_idx = fixed_indices[0] # Usiamo la stessa prima immagine usata per gli altri plot
_, gt_emp = test_dataset[test_idx]

deg_op_emp = DegradationOperator(sr_factor=sr_emp, noise_std=noise_emp, device=device)
deg_op_emp.seed = 42 + test_idx
y_lr_emp = deg_op_emp(gt_emp.to(device)).to(device)

# Inizializziamo DiffPIR
from diffusers import UNet2DModel
diff_unet_emp = UNet2DModel.from_pretrained("google/ddpm-ema-church-256").to(device)
diffpir_emp = DiffPIR(sr_factor=sr_emp, noise_std=noise_emp, sampling_steps=50, device=device, pretrained_unet=diff_unet_emp)

# Test con 3 seed diversi
seeds = [10, 42, 99]
results_emp = []

for s in seeds:
    print(f"Esecuzione DiffPIR con seed={s}...")
    torch.manual_seed(s)
    np.random.seed(s)
    
    with torch.no_grad():
        # Passiamo sempre la stessa y_lr_emp!
        x_res = diffpir_emp.restore(y_lr_emp).cpu()
        results_emp.append(x_res)

# Plotting
fig_emp, axes_emp = plt.subplots(1, 5, figsize=(20, 5))
axes_emp[0].imshow(np.clip(gt_emp.permute(1,2,0).cpu().numpy(), 0, 1))
axes_emp[0].set_title("Ground Truth", fontsize=14)
axes_emp[0].axis("off")

y_up_emp = torch.nn.functional.interpolate(y_lr_emp.unsqueeze(0), size=(256, 256), mode='bicubic', align_corners=False).squeeze(0).cpu()
axes_emp[1].imshow(np.clip(y_up_emp.permute(1,2,0).numpy(), 0, 1))
axes_emp[1].set_title(f"Degraded (SR={sr_emp}, n={noise_emp})", fontsize=14)
axes_emp[1].axis("off")

for i, (s, res) in enumerate(zip(seeds, results_emp)):
    axes_emp[i+2].imshow(np.clip(res.permute(1,2,0).numpy(), 0, 1))
    axes_emp[i+2].set_title(f"DiffPIR (Seed={s})", fontsize=14)
    axes_emp[i+2].axis("off")

plt.tight_layout()
plt.savefig("empirical_test_hallucination.png", dpi=150, bbox_inches='tight')
plt.show()
print("Test completato! Immagine salvata come 'empirical_test_hallucination.png'")
